#### Offside detection demo (SAM2 + pitch calibration)

Run cells top to bottom. A PyQt5 window opens for each step.

**Workflow (4 steps)**
1. Left-click **2 points** on a grass stripe boundary → Enter
2. Left-click **1 point** toward the goal → Enter
3. Left-click **attacking player** (full body) → Enter (SAM2 mask)
4. Left-click **last defender** (full body) → Enter (SAM2 mask)

Verdict: compare the **foremost mask pixel toward goal** on each player (any body part — not feet only).

Picker: **Left** = positive, **Right** = negative, **Enter/Esc** = done, **Backspace** = undo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as IPImage, display
from PIL import Image

from demo import (
    OUTPUT_DIR,
    analyze_masks,
    build_pitch_frame,
    mask_centroid,
    pick_calibration_points,
    print_verdict,
    visualize,
)
from sam2_helpers import load_sam2, segment_player

ROOT = Path(".").resolve()
IMAGE_PATH = ROOT / "imgs" / "offside.png"

img_pil = Image.open(IMAGE_PATH).convert("RGB")
img_np = np.array(img_pil)
print(f"Image: {IMAGE_PATH}  shape={img_np.shape}")

plt.figure(figsize=(10, 6))
plt.imshow(img_np)
plt.title("Input frame")
plt.axis("off")
plt.show()

In [ ]:
model, processor, device = load_sam2()

In [ ]:
s1, s2, goal_pt = pick_calibration_points(img_np)
print(f"Stripe clicks: {s1} -> {s2}")
print(f"Goal hint:     {goal_pt}")

In [ ]:
attacker_mask, attacker_pos, _ = segment_player(
    model, processor, device, img_pil, img_np,
    "3) Attacking player (pass receiver)",
)
print(f"Attacker mask pixels: {attacker_mask.sum()}")

In [ ]:
defender_mask, defender_pos, _ = segment_player(
    model, processor, device, img_pil, img_np,
    "4) Last defender",
)
print(f"Defender mask pixels: {defender_mask.sum()}")

In [ ]:
frame = build_pitch_frame(s1, s2, goal_pt, mask_centroid(defender_mask))
result = analyze_masks(attacker_mask, defender_mask, frame, s1, s2)
print_verdict(result)

In [ ]:
out_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_offside.png"
visualize(img_np, result, attacker_mask, defender_mask, out_path, show=True)
display(IPImage(filename=str(out_path)))